---
title: "3.5.d Escenarios: fuego, sequía y punto de no retorno"
short_title: "d. Escenarios"
subtitle: "Proyecciones de condiciones secas y cálidas que favorecen el fuego"
description: "Escenarios CMIP6 para las condiciones meteorológicas de incendio en Leticia y riesgo de transición del bosque."
keywords: [incendios, CMIP6, punto de no retorno, deforestación, Amazonía]
---

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mromerot/leticia/blob/main/book/03_analisis/35_incendios/d_escenarios.ipynb)

In [1]:
# Preparación del entorno. En Colab instala el paquete del proyecto desde GitHub (1–2 minutos);
# en una copia local usa el paquete ya instalado con `pip install -e .`.
import sys, subprocess
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "leticia @ git+https://github.com/mromerot/leticia"], check=True)
from leticia.libro import preparar, intentar
preparar()

El riesgo de incendio en el futuro depende de dos fuerzas: el **clima**, que hará las estaciones secas más cálidas
y con mayor déficit hídrico, y el **uso del suelo**, que decide cuánto bosque se abre y cuánta ignición hay.
Flores et al. estiman que hacia 2050 entre el 10 % y el 47 % del bosque amazónico estará expuesto a
perturbaciones combinadas que podrían desencadenar transiciones del ecosistema [@flores2024].

In [2]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.patches import Patch

from leticia import aoi, clima, espacial, exposicion, fuentes, riesgo, viz
from leticia.config import parametros

pi = parametros()["incendios"]
urbana = aoi.area_urbana()
mun = fuentes.limite_municipal()


def focos(sensor="modis", anios=None):
    f = fuentes.focos_calor(sensor, anios)
    if sensor == "modis":
        f = f[f["confidence"] >= pi["confianza_modis_min"]]
    else:
        f = f[f["confidence"].astype(str).str.lower().isin(pi["confianza_viirs"])]
    f = f[f["type"] == 0] if "type" in f else f          # solo vegetación (excluye fuentes estáticas y volcanes)
    return gpd.GeoDataFrame(f, geometry=gpd.points_from_xy(f.longitude, f.latitude), crs=4326)

## 1. Condiciones que favorecen el fuego (CCKP)

In [3]:
VARS = {"tasmax": "temperatura máxima (°C)", "cdd": "máx. días secos consecutivos", "spei12": "SPEI-12", "hd35": "días con Tmax > 35 °C"}
tabla = fuentes.cckp_tabla(list(VARS))
med = tabla[tabla.percentil == "median"].pivot_table(index="variable", columns=["periodo", "escenario"], values="valor")
med.columns = [f"{p} {viz.NOMBRE_ESC[e]}" for p, e in med.columns]
med.index = [VARS[v] for v in med.index]
display(Markdown("**Cambio proyectado respecto a 1995–2014 (mediana del ensamble):**"))
display(med.round(2))

**Cambio proyectado respecto a 1995–2014 (mediana del ensamble):**

,2040-2059 SSP1-2.6,2040-2059 SSP2-4.5,2040-2059 SSP5-8.5,2080-2099 SSP1-2.6,2080-2099 SSP2-4.5,2080-2099 SSP5-8.5
máx. días secos consecutivos,0.75,0.70,0.80,0.60,1.05,2.15
días con Tmax > 35 °C,3.15,4.95,19.60,4.20,26.05,152.70
SPEI-12,-0.43,-0.62,-0.93,-0.50,-0.90,-2.00
temperatura máxima (°C),1.04,1.24,1.78,1.09,2.18,4.83


In [4]:
viz.pie(viz.nota_fuente("proyecciones_cckp"))

*Fuente: Banco Mundial – Climate Change Knowledge Portal (CCKP) — CMIP6 0,25° (reducción de escala estadística), SSP1-2.6, SSP2-4.5, SSP5-8.5.*

## 2. Escenarios de uso del suelo alrededor de la ciudad

In [5]:
loss = fuentes.bosque_hansen("lossyear")
tc = fuentes.bosque_hansen("treecover2000")
b = urbana.to_crs(4326).buffer(0.05).intersection(mun.geometry.iloc[0])   # ~5 km alrededor del área urbana, lado colombiano
l = loss.rio.write_crs(4326).rio.clip(b.geometry, drop=True)
t = tc.rio.write_crs(4326).rio.clip(b.geometry, drop=True)
px_ha = (0.00025 * 111320) ** 2 * np.cos(np.radians(4.2)) / 1e4
bosque2000 = float(((t.values >= 50) & np.isfinite(t.values)).sum()) * px_ha
perdida = pd.Series(l.values.ravel()).value_counts().drop(0, errors="ignore").sort_index() * px_ha
perdida.index = perdida.index.astype(int) + 2000
tasa = perdida.loc[2015:].mean()
restante = bosque2000 - perdida.sum()
filas = [{"escenario": "Tendencia 2015–2024", "pérdida anual (ha)": tasa, "bosque en 2050 (ha)": restante - tasa * 26},
         {"escenario": "Duplicación de la tasa", "pérdida anual (ha)": 2 * tasa, "bosque en 2050 (ha)": restante - 2 * tasa * 26},
         {"escenario": "Deforestación cero desde 2030 (Pacto de Leticia)", "pérdida anual (ha)": tasa,
          "bosque en 2050 (ha)": restante - tasa * 6}]
display(Markdown(f"Bosque (cobertura ≥ 50 %) en 2000 en un radio de ~5 km de la ciudad (lado colombiano): **{bosque2000:,.0f} ha**; perdido 2001–2024: "
                 f"{perdida.sum():,.0f} ha.".replace(",", ".")))
display(pd.DataFrame(filas).set_index("escenario").round(0))

Bosque (cobertura ≥ 50 %) en 2000 en un radio de ~5 km de la ciudad (lado colombiano): **8.443 ha**; perdido 2001–2024: 1.494 ha.

,pérdida anual (ha),bosque en 2050 (ha)
escenario,,
Tendencia 2015–2024,75.00,"5,001.00"
Duplicación de la tasa,150.00,"3,053.00"
Deforestación cero desde 2030 (Pacto de Leticia),75.00,"6,499.00"


**Lectura.** Los escenarios de uso del suelo son simples extrapolaciones de la tasa reciente de pérdida de bosque.
Su mensaje es de política: el **Pacto de Leticia por la Amazonía** (2019) y la Declaración de Belém (2023)
comprometen a los países amazónicos con la reducción de la deforestación. Para Leticia, conservar el bosque
periurbano reduce a la vez el riesgo de incendio, el calor urbano ([3.4](../34_calor/index.md)) y la presión sobre
las fuentes de agua.